In [ ]:
import pandas as pd

from gsm_benchmarker.results_analyser.utils import pandas_to_latex

from results_notebook_setup import load_results


In [ ]:
results_loader = load_results(n_boot=None)
full_results = results_loader.full_results

In [ ]:
df = results_loader.gsm.mres.get_glmm1_data(variant='main', metric='correct')

# per-item, per-model base-vs-variant accuracy
item_compare = (
    df.groupby(['model', 'id', 'is_variant'])['is_correct']
    .mean()
    .unstack('is_variant')
    .rename(columns={0: 'acc_variant_0', 1: 'acc_variant_1'})
)
item_compare['diff'] = item_compare['acc_variant_1'] - item_compare['acc_variant_0']

# collapse to one row per model
item_compare_summary = (
    item_compare['diff']
    .groupby('model')
    .agg(
        mean_diff='mean',
        prop_large_split=lambda s: (s.abs() > 0.8).mean(),
        n_items='size',
    )
)

item_compare_summary

In [ ]:
base_variant_accuracies = df.groupby(['model', 'is_variant'])['is_correct'].mean().unstack('is_variant')

grp = df.groupby(['id', 'model'])['is_correct'].agg(['size', 'mean'])
grp['constant'] = grp['mean'].isin([0, 1])
grp['constant'].mean()  # proportion of degenerate groups


def perc_fmt(precision):
    def wrapper(x):
        px = 100 * x
        if precision == 0:
            return str(int(round(px)))
        return f"{px:.{precision}f}"
    return wrapper

separation_summary_df = pd.DataFrame({
    'GSM-Base acc.': base_variant_accuracies[0].apply(perc_fmt(1)),
    'GSM-Variants acc.': base_variant_accuracies[1].apply(perc_fmt(2)),
    'Mean acc. diff.': item_compare_summary.mean_diff.apply(perc_fmt(2)),
    'Large split': item_compare_summary.prop_large_split.apply(perc_fmt(0)),
    'Constancy': grp.groupby('model').constant.mean().apply(perc_fmt(0)),
}).sort_values('Constancy', ascending=False)

separation_summary_df

In [ ]:

print(pandas_to_latex(
    separation_summary_df,
    position="H",
    caption=r"Summary of per-model marginal separation and cluster-level degeneracy check. All values given in \%. ``Constancy'' is the proportion of items with degenerate accuracy (all correct or all incorrect for a given template id, pooled across Variants/Base datasets). ``Large split'' is the proportion of items with a difference in accuracy between base and variant greater than 80 percentage points.",
))